# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata  # single object
print(f"Loaded dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and fields using @id
print('Available record sets:')
rs_overview = []
for rs in getattr(metadata, 'record_sets', []):
    print(f"- RecordSet name: {getattr(rs, 'name', '')}, @id: {getattr(rs, '@id', None)}")
    rs_overview.append({
        'name': getattr(rs, 'name', ''),
        '@id': getattr(rs, '@id', None),
        'fields': getattr(rs, 'fields', [])
    })
    for fld in getattr(rs, 'fields', []):
        print(f"    - Field: {getattr(fld, 'name', '')}, @id: {getattr(fld, '@id', None)}")
if not rs_overview:
    print('No record sets are embedded directly; checking for those defined in files/distribution.')

# If record_sets is empty, look for dataset.distribution with record_set info
if not getattr(metadata, 'record_sets', []):
    distributions = getattr(metadata, 'distribution', [])
    for dist in distributions:
        record_sets = getattr(dist, 'record_sets', []) if hasattr(dist, 'record_sets') else []
        if record_sets:
            for rs in record_sets:
                print(f"- RecordSet name: {getattr(rs, 'name', '')}, @id: {getattr(rs, '@id', None)}")
                for fld in getattr(rs, 'fields', []):
                    print(f"    - Field: {getattr(fld, 'name', '')}, @id: {getattr(fld, '@id', None)}")
        else:
            # Some Croissant files define default record sets per data file
            print(f"Distribution with @id: {getattr(dist, '@id', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# For FAIR², the main data is often in a record set tied to the main file.

# Let's enumerate all record set @id values:
available_record_set_ids = []
if getattr(metadata, 'record_sets', []):
    available_record_set_ids = [getattr(rs, '@id', None) for rs in metadata.record_sets]
else:
    for dist in getattr(metadata, 'distribution', []):
        if hasattr(dist, 'record_sets'): # Not all schemas store these per distribution
            available_record_set_ids += [getattr(rs, '@id', None) for rs in dist.record_sets]

# If no explicit record sets, mlcroissant can usually infer them from the main file object (most often @id==distribution[0] record_set)
if not available_record_set_ids:
    # Try default record set id, which is usually the distribution @id
    available_record_set_ids = [getattr(metadata, 'distribution', [])[0]['@id']] if metadata and getattr(metadata, 'distribution', False) else []

print('RecordSet @id list:', available_record_set_ids)

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in available_record_set_ids:
    print(f"Loading records for RecordSet @id={record_set_id} ...")
    try:
        records = list(ds.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with shape {df.shape}: columns={df.columns.tolist()[:8]}{'...' if len(df.columns)>8 else ''}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the first usable record set id for the next steps
main_record_set = available_record_set_ids[0] if available_record_set_ids else None
if main_record_set:
    print(f"Columns in main record set ({main_record_set}):\n{dataframes[main_record_set].columns.tolist()}")
    display(dataframes[main_record_set].head())
else:
    print('No record set found to extract data.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field and a group field for EDA. This will depend on the columns parsed above.
if main_record_set:
    df = dataframes[main_record_set]
    print('Available columns:', df.columns.tolist())
    
    # Example: Try using 'Normalized Abundance' or 'MW' ('Molecular Weight') if they exist
    possible_numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ['abunda', 'mw', 'molecular', 'intens', 'count', 'coverage', 'area'])]
    print('Detected possible numeric fields:', possible_numeric_fields)
    
    # Select one
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns)>0 else df.columns[0]
    print('Chosen numeric_field for analysis:', numeric_field)
    
    # Filtering step
    # Work with NaNs: Cast to numeric as needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].dtype=='float' else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalization
    mu, sigma = filtered_df[numeric_field].mean(), filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Grouping step (e.g., by 'Description' or other categorical feature)
    possible_group_fields = [c for c in df.columns if any(s in c.lower() for s in ['desc', 'sample', 'condition', 'type', 'group'])]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean()
        print(f"Grouped data by {group_field} - mean of {numeric_field}:")
        display(grouped_df.head())
        # Save for visualization
        grouped_to_plot = grouped_df.sort_values(ascending=False).head(10)
    else:
        grouped_to_plot = filtered_df[[numeric_field, f"{numeric_field}_normalized"]].copy()
else:
    print('No main record set found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_record_set:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Bar plot of top groups if grouping was possible
    if 'grouped_to_plot' in locals() and group_field:
        plt.figure(figsize=(10,4))
        grouped_to_plot.plot(kind='bar', color='skyblue')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
    
    # Scatter plot (if another numeric field exists)
    numeric_fields_rest = [c for c in possible_numeric_fields if c!=numeric_field]
    if numeric_fields_rest:
        yfield = numeric_fields_rest[0]
        filtered_df[yfield] = pd.to_numeric(filtered_df[yfield], errors='coerce')
        plt.figure(figsize=(7,5))
        plt.scatter(filtered_df[numeric_field], filtered_df[yfield], alpha=0.5)
        plt.xlabel(numeric_field)
        plt.ylabel(yfield)
        plt.title(f"Scatter plot: {numeric_field} vs. {yfield}")
        plt.show()
else:
    print('No visualizations available. No main dataframe.')

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using the Croissant schema and explored one or more main record sets identified by their `@id`. We reviewed available fields, performed numeric filtering and normalization, and visualized both distributions and simple group summaries, using only entity references by their `@id`. This approach enables transparent tracking and reproducible FAIR data analysis.

Further work could include more domain-specific feature engineering, advanced visualizations, or linking record set fields to biological annotation data for enhanced interpretation.